In [ ]:
import numpy as np
import os
import cv2
import random
import copy
import matplotlib.pyplot as plt

import torch
from model import *
from data import *
from torchvision import models as tv_models



seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
cv2.setRNGSeed(seed)
def worker_init_fn(worker_id):
    worker_seed = seed + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
torch.use_deterministic_algorithms(True, warn_only=True)
g = torch.Generator()
g.manual_seed(seed)

#====================================================================================================================================

IMAGE_SIZE = 256
NUM_WORKERS = 4
PIN_MEMORY = True
LABELED_RATIO = 0.1
NAME_DATASET = 'OTU'


BATCH_SIZE = 8


In [ ]:
class ConvBlock(nn.Module):
    """Two (Conv -> BN -> ReLU) layers, same padding."""
    def __init__(self, in_ch, out_ch, k=3):
        super().__init__()
        p = k // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, k, padding=p), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, k, padding=p), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class VGG16Encoder(nn.Module):
    """VGG16 (ImageNet) feature extractor returning 4 skip connections + bottleneck.

    Skip channels = (64, 128, 256, 512), bottleneck = (512, H/16, W/16).
    Replicates Keras' block1_conv2 / block2_conv2 / block3_conv3 / block4_conv3 / block5_conv3.
    """
    def __init__(self, pretrained=True):
        super().__init__()
        vgg = tv_models.vgg16(weights=tv_models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
        feats = vgg.features
        # VGG16 features split (indices match torchvision impl):
        #   block1: 0..3   (conv,relu,conv,relu)  -> idx 3 is block1_conv2 ReLU
        #   pool   : 4
        #   block2: 5..8   -> idx 8  = block2_conv2 ReLU
        #   pool   : 9
        #   block3: 10..15 -> idx 15 = block3_conv3 ReLU
        #   pool   : 16
        #   block4: 17..22 -> idx 22 = block4_conv3 ReLU
        #   pool   : 23
        #   block5: 24..29 -> idx 29 = block5_conv3 ReLU  (bottleneck)
        self.b1 = nn.Sequential(*feats[ 0: 4])
        self.p1 = feats[ 4]
        self.b2 = nn.Sequential(*feats[ 5: 9])
        self.p2 = feats[ 9]
        self.b3 = nn.Sequential(*feats[10:16])
        self.p3 = feats[16]
        self.b4 = nn.Sequential(*feats[17:23])
        self.p4 = feats[23]
        self.b5 = nn.Sequential(*feats[24:30])
    def forward(self, x):
        s1 = self.b1(x)
        s2 = self.b2(self.p1(s1))
        s3 = self.b3(self.p2(s2))
        s4 = self.b4(self.p3(s3))
        b  = self.b5(self.p4(s4))
        return b, [s1, s2, s3, s4]


class DecoderBlockNormal(nn.Module):
    """Upsample (bilinear) -> concat skip -> ConvBlock."""
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.conv = ConvBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class VGG16UNet(nn.Module):
    """VGG16 -> SPPFast -> normal decoder (paper uses decoder_branch_normal) -> 1x1 sigmoid."""
    def __init__(self, num_classes=1, pretrained_encoder=True):
        super().__init__()
        self.encoder = VGG16Encoder(pretrained=pretrained_encoder)
        # Skip channels: s1=64, s2=128, s3=256, s4=512
        self.dec4 = DecoderBlockNormal(in_ch=512, skip_ch=512, out_ch=512)
        self.dec3 = DecoderBlockNormal(in_ch=512, skip_ch=256, out_ch=256)
        self.dec2 = DecoderBlockNormal(in_ch=256, skip_ch=128, out_ch=128)
        self.dec1 = DecoderBlockNormal(in_ch=128, skip_ch= 64, out_ch= 64)
        self.out_conv = nn.Conv2d(64, num_classes, 1)
    def forward(self, x):
        b, [s1, s2, s3, s4] = self.encoder(x)
        d = self.dec4(b, s4)
        d = self.dec3(d, s3)
        d = self.dec2(d, s2)
        d = self.dec1(d, s1)
        return torch.sigmoid(self.out_conv(d))

In [ ]:
models = [MT_Proposed(with_vgg16bn=True), MT_Proposed(with_vgg16bn=True), MT_Proposed(with_vgg16bn=True), MT_Proposed(with_vgg16bn=True)]
model_names = ['C=50', 'C=100', 'C=200', 'C=400']


checkpoints = ['weight/proposed-b8-c50-k0.01-BankWeightExp-IMBLoss.pth', 'weight/proposed-b8-c100-k0.01-BankWeightExp-IMBLoss.pth', 
               'weight/proposed-b8-c200-k0.01-BankWeightExp-IMBLoss.pth', 'weight/proposed-b8-c400-k0.01-BankWeightExp-IMBLoss.pth']

dataset = get_datasets(name='OTU', with_cls=True)

In [ ]:
FONTSIZE = 20

plt.rcParams.update({
    'font.size': FONTSIZE,
    'font.weight': 'bold',
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold'
})

def plot1(
    images_dict,
    masks_dict,
    models,
    checkpoints,
    model_names,
    device='cuda:0',
    threshold=0.5
):

    import torch
    import matplotlib.pyplot as plt

    M = len(models)

    # load weights
    for model, ckpt in zip(models, checkpoints):
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state)
        model.to(device)
        model.eval()

    # gom toàn bộ sample hợp lệ
    samples = []

    for cls_id in sorted(images_dict.keys()):

        if images_dict[cls_id] is None:
            continue

        image = images_dict[cls_id].to(device)   # (1,C,H,W)
        mask = masks_dict[cls_id]                # (1,1,H,W)

        samples.append((cls_id, image, mask))

    B = len(samples)

    # forward từng model
    all_preds = []

    with torch.no_grad():

        for model, model_name in zip(models, model_names):

            model_preds = []

            for cls_id, image, _ in samples:

                if 'Proposed' in model_name or 'Ours' in model_name or 'proposed' in model_name or \
                      'only' in model_name or 'C=' in model_name:
                    out, _, _ = model(image)
                else:
                    out = model(image)

                pred = (out > threshold).float()
                model_preds.append(pred.cpu())

            all_preds.append(model_preds)

    # ======================================================
    # plot
    # ======================================================
    fig, axes = plt.subplots(
        B,
        M,
        figsize=(4 * M, 4 * B),
        gridspec_kw={
            'wspace': 0.0,   # giảm khoảng cách ngang
            'hspace': 0.0    # giảm khoảng cách dọc
        }
    )

    # xử lý khi chỉ có 1 row hoặc 1 column
    if B == 1:
        axes = axes[None, :]

    if M == 1:
        axes = axes[:, None]

    for i, (cls_id, image, mask) in enumerate(samples):

        img_np = image[0].permute(1, 2, 0).detach().cpu().numpy()
        gt = mask[0].squeeze().cpu().numpy()

        for j in range(M):

            pred = all_preds[j][i][0].squeeze().numpy()

            ax = axes[i, j]

            ax.imshow(img_np)

            # GT contour
            ax.contour(
                gt,
                levels=[0.5],
                colors='green',
                linewidths=4
            )

            # Prediction contour
            ax.contour(
                pred,
                levels=[0.5],
                colors='red',
                linewidths=4
            )

            ax.axis('off')

            # title
            if i == 0:
                ax.set_title(
                    model_names[j],
                    fontsize=FONTSIZE,
                    pad=1
                )

            # label class bên trái
            if j == 0:
                ax.set_ylabel(
                    f'Class {cls_id}',
                    fontsize=FONTSIZE,
                    rotation=90,
                    labelpad=5,
                    va='center'
                )

    # giảm tối đa padding ngoài figure
    fig.subplots_adjust(
        left=0.0,
        right=0.01,
        top=0.0,
        bottom=0.0,
        wspace=0.2,
        hspace=0.2
    )

    plt.show()

def plot2(
    dataset,
    models,
    checkpoints,
    model_names,
    device='cuda:0',
    threshold=0.5,
    num_per_class=5,
    num_classes=8
):

    M = len(models)

    # ==========================================================
    # load weights
    # ==========================================================
    for model, ckpt in zip(models, checkpoints):

        state = torch.load(ckpt, map_location=device)

        model.load_state_dict(state)
        model.to(device)
        model.eval()

    # ==========================================================
    # lấy sample theo từng class
    # ==========================================================
    class_samples = {i: [] for i in range(num_classes)}

    for idx in range(len(dataset)):

        image, _, mask, _, cls = dataset[idx]

        cls = cls.item()

        if len(class_samples[cls]) < num_per_class:

            class_samples[cls].append((
                image.unsqueeze(0),
                mask.unsqueeze(0)
            ))

        done = True

        for c in range(num_classes):
            if len(class_samples[c]) < num_per_class:
                done = False
                break

        if done:
            break

    # ==========================================================
    # plot theo từng class
    # ==========================================================
    for cls_id in range(num_classes):

        # if cls_id != 7:
        #     continue

        samples = class_samples[cls_id]

        B = len(samples)

        # ======================================================
        # forward
        # ======================================================
        all_preds = []

        with torch.no_grad():

            for model, model_name in zip(models, model_names):

                model_preds = []

                for image, _ in samples:

                    image = image.to(device)

                    if 'Proposed' in model_name or 'Ours' in model_name or 'proposed' in model_name or \
                      'only' in model_name or 'C=' in model_name:
                        out, _, _ = model(image)
                    else:
                        out = model(image)

                    pred = (out > threshold).float()

                    model_preds.append(pred.cpu())

                all_preds.append(model_preds)

        # ======================================================
        # figure cho riêng class hiện tại
        # ======================================================
        fig, axes = plt.subplots(
            B,
            M,
            figsize=(5 * M, 4 * B)
        )

        # nếu chỉ có 1 sample
        if B == 1:
            axes = axes[None, :]

        fig.suptitle(
            f'Class {cls_id}',
            fontsize=FONTSIZE,
            y=1.02
        )

        for i in range(B):

            image, mask = samples[i]

            img_np = image[0].permute(1, 2, 0).cpu().numpy()
            gt = mask[0].squeeze().cpu().numpy()

            for j in range(M):

                ax = axes[i, j]

                pred = all_preds[j][i][0].squeeze().numpy()

                ax.imshow(img_np)

                # GT contour
                ax.contour(
                    gt,
                    levels=[0.5],
                    colors='green',
                    linewidths=4
                )

                # Prediction contour
                ax.contour(
                    pred,
                    levels=[0.5],
                    colors='red',
                    linewidths=4
                )

                ax.axis('off')

                # title
                if i == 0:
                    ax.set_title(
                        model_names[j],
                        fontsize=FONTSIZE,
                    )

                # sample label
                if j == 0:

                    ax.set_ylabel(
                        f'Sample {i + 1}',
                        fontsize=FONTSIZE,
                        rotation=0,
                        labelpad=50,
                        va='center'
                    )

        plt.tight_layout()
        fig.subplots_adjust(
            left=0.01,
            right=0.99,
            top=0.92,
            bottom=0.01,
            wspace=0.01,
            hspace=0.01
        )
        plt.show()

In [ ]:
dataset = get_datasets(name='OTU', with_cls=True)

In [ ]:
images = {i: None for i in range(8)}
masks = {i: None for i in range(8)}

# ==========================================================
# in toàn bộ index theo từng class
# ==========================================================
class_indices = {i: [] for i in range(8)}

for idx in range(len(dataset)):

    _, _, _, _, cls = dataset[idx]

    cls = cls.item()

    class_indices[cls].append(idx)

# for cls_id in range(8):

#     print(f'Class {cls_id}:')
#     print(class_indices[cls_id])
#     print()

# ==========================================================
# chọn index mong muốn cho từng class
# ==========================================================
selected_indices = {
    0: 419,
    2: 11,
    3: 73,
    4: 26,
}

# ==========================================================
# load sample theo selected index
# ==========================================================
for cls_id, idx in selected_indices.items():

    image, _, mask, _, cls = dataset[idx]

    images[cls_id] = image.unsqueeze(0)
    masks[cls_id] = mask.unsqueeze(0)

plot1(images, masks, models, checkpoints, model_names, device='cuda')

In [ ]:
# ==========================================================
# in toàn bộ index theo từng class
# ==========================================================
class_indices = {i: [] for i in range(8)}

for idx in range(len(dataset)):

    _, _, _, _, cls = dataset[idx]

    cls = cls.item()

    class_indices[cls].append(idx)

for cls_id in range(8):

    print(f'Class {cls_id}:')
    print(class_indices[cls_id])
    print()

In [ ]:
# images = {i: None for i in range(8)}
# masks = {i: None for i in range(8)}

# # ==========================================================
# # in toàn bộ index theo từng class
# # ==========================================================
# class_indices = {i: [] for i in range(8)}

# for idx in range(len(dataset)):

#     _, _, _, _, cls = dataset[idx]

#     cls = cls.item()

#     class_indices[cls].append(idx)

# # for cls_id in range(8):

# #     print(f'Class {cls_id}:')
# #     print(class_indices[cls_id])
# #     print()

# # ==========================================================
# # chọn index mong muốn cho từng class
# # ==========================================================
# selected_indices = {
#     0: 47,
#     1: 14,
#     2: 4,
#     3: 246,
#     4: 46,
#     5: 36,
#     6: 78,
#     7: 58
# }

# # ==========================================================
# # load sample theo selected index
# # ==========================================================
# for cls_id, idx in selected_indices.items():

#     image, _, mask, _, cls = dataset[idx]

#     images[cls_id] = image.unsqueeze(0)
#     masks[cls_id] = mask.unsqueeze(0)

# plot1(images, masks, models, checkpoints, model_names, device='cuda')

In [ ]:
# plot2(
#     dataset=dataset,
#     models=models,
#     checkpoints=checkpoints,
#     model_names=model_names,
#     device='cuda',
#     num_per_class=10
# )